In [8]:
%pip install anthropic python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
from dotenv import load_dotenv

load_dotenv()

True

In [10]:
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-5"

In [11]:
def add_user_message(messages, text):
    user_message = { "role": "user", "content": text }
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = { "role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model" : model,
        "max_tokens" : 1000,
        "messages" : messages,
        "temperature" : temperature
    }

    if system:
        params["system"] = system

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    message = client.messages.create(**params)
    return message.content[0].text

In [12]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)


In [13]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [ ]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [15]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    score = 10

    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [16]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

        return result

In [18]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

    results = run_eval(dataset)

In [19]:
print(json.dumps(results, indent=2))

{
  "output": "```python\ndef parse_arn(arn):\n    \"\"\"\n    Parses an AWS ARN (Amazon Resource Name) and returns a dictionary with its components.\n    \n    Args:\n        arn (str): The ARN string to parse\n        \n    Returns:\n        dict: A dictionary containing the ARN components:\n            - partition: The partition (e.g., 'aws', 'aws-cn', 'aws-us-gov')\n            - service: The service name (e.g., 's3', 'ec2', 'iam')\n            - region: The region (can be empty for global services)\n            - account_id: The account ID (can be empty for some services)\n            - resource: The resource identifier\n            \n    Raises:\n        ValueError: If the ARN format is invalid\n    \"\"\"\n    if not arn or not isinstance(arn, str):\n        raise ValueError(\"ARN must be a non-empty string\")\n    \n    # ARN format: arn:partition:service:region:account-id:resource\n    parts = arn.split(':', 5)  # Split into max 6 parts\n    \n    if len(parts) < 6:\n        r